## Tydzień 2 Dzień 2 - Orkiestracja

Nasz pierwszy projekt we frameworku agentowym!!

### Część 1: Konfiguracja e-maila

### Część 2: Orkiestracja przez kod

### Część 3: Orkiestracja przez LLM

- 3a: przez narzędzia
- 3b: przez handoffs

**Ta wersja jest inna niż oryginał: zamiast frameworka OpenAI Agents SDK, budujemy dokładnie te same mechanizmy orkiestracji ręcznie, na natywnym Anthropic SDK.** Tak jak w `1_lab1.pl.ipynb`, żadna z abstrakcji tego labu (`Agent`, `Runner`, `handoffs`, `agent.as_tool()`, `draw_graph()`, `ModelSettings`) nie ma odpowiednika w Anthropic SDK - to framework zbudowany na OpenAI Responses API, nie sam interfejs API. Ten notatnik redefiniuje lokalnie `run()` i `trace()` z `1_lab1.pl.ipynb` (każdy notatnik tego kursu jest samodzielnym projektem), rozszerzając je o trzy nowe mechanizmy potrzebne do orkiestracji: klienta asynchronicznego (`AsyncAnthropic`) do prawdziwie równoległych wywołań (`asyncio.gather`), wzorzec "agent jako narzędzie" (subagent wywoływany jako zwykłe narzędzie, którego wynik wraca do agenta nadrzędnego) i uproszczony odpowiednik handoffs (subagent przejmuje kontrolę na stałe, jego wynik kończy cały `run()`). `draw_graph()` (wizualizacja przez graphviz) zastępujemy prostym, tekstowym drzewem w konsoli - bez dorzucania nowej zależności dla jednej wizualizacji.

## Część 1: Konfiguracja e-maila

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">WAŻNE, PRZECZYTAJ - Wysyłanie e-maili</h2>
            <span style="color:#ff7800;">Napiszemy agenta, który wysyła e-maile. Najlepszym sposobem jest użycie dostawcy e-mail,
            takiego jak SendGrid albo Resend. Ale to wymaga sporo pracy: trzeba wysyłać z odpowiedniego hosta pocztowego, gdzie posiadasz domenę i możesz to udowodnić rekordami DNS. To spory kłopot.<br/><br/>
            Dlatego zrobimy to darmowym i prostym sposobem: konfiguracją serwera SMTP, żeby wysyłać bezpośrednio z Twojej skrzynki. To łatwa konfiguracja, ale mniej potężna (np. nie możesz odbierać e-maili). Jeśli chcesz pójść dalej, użyj zamiast tego SendGrid albo Resend..<br/><br/>
            A tak naprawdę wysyłka e-maila jest opcjonalna. Robimy to, żeby pokazać Agenta wysyłającego e-maile. Ważna jest praca Agenta, nie sama wysyłka. Możesz śmiało zamienić tę funkcję na powiadomienie Pushover, jeśli wolisz.
            </span>
        </td>
    </tr>
</table>

## Konfiguracja wysyłki e-maili przez Twój serwer SMTP

### KROK 1: Znajdź swój serwer SMTP

Wygoogluj albo zapytaj ChatGPT / Claude o serwer SMTP dla swojej poczty. Oto kilka popularnych. Niektórzy dostawcy poczty mogą mieć wyłączone serwery SMTP (np. Microsoft 365 dla kont firmowych/szkolnych).

Google: smtp.gmail.com
Outlook.com / Hotmail / Live: smtp-mail.outlook.com
Microsoft 365: smtp.office365.com
iCloud Mail: smtp.mail.me.com

Dodaj do pliku .env:

`EMAIL_SMTP_SERVER=xxxx`

### KROK 2: Zdobądź hasło aplikacji

Wygoogluj, jak to zrobić dla swojego dostawcy poczty. Dla Gmaila musisz mieć włączoną weryfikację dwuetapową. Potem wejdź na tę stronę:

https://myaccount.google.com/apppasswords

Nadaj dowolną nazwę; skopiuj hasło i dodaj je do pliku .env, usuwając spacje, które tam dokłada. (Powinno mieć 16 znaków, bez spacji.)

`EMAIL_APP_PASSWORD=xxxx`

### KROK 3: Dodaj swój adres e-mail:

`EMAIL_ADDRESS=xxx`

Pamiętaj, żeby zapisać plik .env!

In [ ]:
# Importy tej wersji notatnika. Zamiast biblioteki agents (OpenAI Agents SDK) importujemy wyłącznie natywny klient Anthropic, w wariancie asynchronicznym.
# AsyncAnthropic jest potrzebny, bo Część 2 tego labu demonstruje orkiestrację równoległą (asyncio.gather) - kilka wywołań LLM naraz, co wymaga klienta z await zamiast synchronicznego.
# contextmanager i time odtwarzają trace() z 1_lab1.pl.ipynb (redefiniowane tutaj, bo każdy notatnik tego kursu jest samodzielnym projektem).
# smtplib i email.message obsługują wysyłkę e-maili przez SMTP - ta część jest niezależna od dostawcy LLM i zostaje bez zmian.
# json przyda się do serializacji wyników narzędzi zwracanych do Claude, tak jak w 1_lab1.pl.ipynb.

from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API, SMTP) z pliku .env
import requests  # wywołania HTTP do Pushover (fallback, gdy e-mail nie jest skonfigurowany)
from anthropic import AsyncAnthropic  # asynchroniczny klient Anthropic - zastępuje framework agents, potrzebny do orkiestracji równoległej
import os  # zmienne środowiskowe
import asyncio  # asyncio.gather do równoległego uruchamiania kilku wywołań run() naraz
import json  # serializacja wyników narzędzi do formatu JSON
import time  # pomiar czasu wewnątrz ręcznego trace()
from contextlib import contextmanager  # dekorator do napisania trace() jako context managera
import smtplib  # wysyłka e-maili przez protokół SMTP
from email.message import EmailMessage  # budowa wiadomości e-mail (temat, treść tekstowa i HTML)

load_dotenv(override=True)  # ładuje .env i nadpisuje istniejące zmienne środowiskowe
anthropic = AsyncAnthropic()  # klucz brany z ANTHROPIC_API_KEY w .env; klient asynchroniczny, bo cały ten notatnik używa run() z await

MODEL = "claude-haiku-4-5"  # najtańszy dostępny model - pułap kosztowy na czas przechodzenia przez kurs

In [ ]:
# Sprawdzenie zmiennych środowiskowych potrzebnych do wysyłki e-maili przez SMTP - logika identyczna jak w oryginale, niezależna od dostawcy LLM.
# EMAIL_ADDRESS to adres, z którego (i na który, w tym demo) wysyłamy e-maile; EMAIL_SMTP_SERVER i EMAIL_APP_PASSWORD to dane logowania do serwera SMTP.
# Komunikaty print() są czytane przez Piotra, więc są po polsku, zgodnie z konwencją repo.
# USE_EMAIL zbiera wynik trzech sprawdzeń w jedną flagę - jeśli którejś zmiennej brakuje, notatnik przełączy się na fallback (Pushover) zamiast e-maila.
# Ta komórka nie wymaga żadnej zmiany przy przejściu z OpenAI na Anthropic - SMTP jest niezależny od dostawcy LLM.

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")  # adres e-mail, z którego i na który wysyłamy wiadomości w tym demo
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")  # adres serwera SMTP danego dostawcy poczty
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")  # hasło aplikacji (nie główne hasło konta) do logowania SMTP

if EMAIL_ADDRESS:  # sprawdź, czy zmienna EMAIL_ADDRESS w ogóle jest ustawiona
    print("Adres e-mail jest ustawiony")  # sanity check przeszedł
else:
    print("Adres e-mail nie jest ustawiony")  # brak zmiennej w .env

if EMAIL_SMTP_SERVER:  # sprawdź, czy zmienna EMAIL_SMTP_SERVER w ogóle jest ustawiona
    print("Serwer SMTP jest ustawiony")  # sanity check przeszedł
else:
    print("Serwer SMTP nie jest ustawiony")  # brak zmiennej w .env

if EMAIL_APP_PASSWORD:  # sprawdź, czy zmienna EMAIL_APP_PASSWORD w ogóle jest ustawiona
    print("Hasło aplikacji jest ustawione")  # sanity check przeszedł
else:
    print("Hasło aplikacji nie jest ustawione")  # brak zmiennej w .env

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD  # flaga: czy mamy komplet danych do wysyłki e-maili

if USE_EMAIL:  # jeśli komplet danych jest ustawiony
    print("E-mail jest skonfigurowany i spróbujemy go użyć")  # użyjemy SMTP
else:
    print("E-mail nie jest skonfigurowany; zamiast tego wyślemy powiadomienia push")  # fallback na Pushover

In [ ]:
# Funkcja wysyłająca e-mail przez SMTP - logika identyczna jak w oryginale, niezależna od dostawcy LLM.
# EmailMessage buduje wiadomość z osobną wersją tekstową i HTML (klient pocztowy wybierze tę, którą potrafi wyświetlić).
# smtplib.SMTP + starttls() nawiązuje szyfrowane połączenie z serwerem SMTP na porcie 587 (standardowy port dla STARTTLS).
# server.login() loguje się hasłem aplikacji (nie głównym hasłem konta), server.send_message() faktycznie wysyła wiadomość.
# Ta funkcja nie wymaga żadnej zmiany przy przejściu z OpenAI na Anthropic - SMTP jest niezależny od dostawcy LLM.

def send_email(subject, text_body, html_body):  # wysyła e-mail z podanym tematem i dwiema wersjami treści
    msg = EmailMessage()  # nowa, pusta wiadomość e-mail
    msg["From"] = EMAIL_ADDRESS  # nadawca (w tym demo ten sam adres co odbiorca)
    msg["To"] = EMAIL_ADDRESS  # odbiorca
    msg["Subject"] = subject  # temat wiadomości
    msg.set_content(text_body)  # wersja tekstowa (fallback dla klientów bez obsługi HTML)
    msg.add_alternative(html_body, subtype="html")  # wersja HTML jako alternatywa

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:  # połączenie z serwerem SMTP na porcie 587
        server.starttls()  # szyfruj połączenie (STARTTLS)
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)  # zaloguj się hasłem aplikacji
        server.send_message(msg)  # wyślij wiadomość

In [ ]:
# Ręczne wywołanie testowe funkcji send_email() zdefiniowanej w komórce wyżej.
# Treść jest przetłumaczona na polski, bo to tekst, który faktycznie przeczytasz w skrzynce jako e-mail testowy.
# To wywołanie nie przechodzi jeszcze przez żadnego agenta ani narzędzie Claude - to zwykłe, bezpośrednie wywołanie funkcji Pythona.
# Służy wyłącznie do sprawdzenia, czy dane SMTP z poprzedniej komórki faktycznie działają.

send_email("Test 123", "Trzymam kciuki..", "<html><body><strong>Trzymam kciuki</strong>..</body></html>")  # e-mail testowy

### Jeśli to nie zadziałało, odkomentuj poniższe, żeby nie używać e-maili

In [ ]:
# USE_EMAIL = False  # odkomentuj, jeśli wysyłka e-mail nie zadziałała - wymusi to użycie Pushover zamiast SMTP

### Nasza strategia zapasowa - wyślij push

In [ ]:
# Ta komórka diagnostyczna sprawdza, czy klucze Pushover są ustawione w .env - to fallback, gdy e-mail nie jest skonfigurowany.
# pushover_user i pushover_token to dane logowania do serwisu powiadomień push Pushover, niezwiązane z Anthropic ani OpenAI.
# Sprawdzenie prefiksu ("u" dla usera, "a" dla tokena) to tylko szybki test sanity, że wkleiłeś właściwy rodzaj klucza.
# Komunikaty wypisywane przez print() są czytane przez Piotra, więc są po polsku, zgodnie z konwencją tego repo.
# Ta komórka nie wymaga żadnej zmiany przy przejściu z OpenAI na Anthropic - Pushover jest niezależny od dostawcy LLM.

pushover_user = os.getenv("PUSHOVER_USER")  # identyfikator użytkownika Pushover z .env
pushover_token = os.getenv("PUSHOVER_TOKEN")  # token aplikacji Pushover z .env
pushover_url = "https://api.pushover.net/1/messages.json"  # endpoint API Pushover do wysyłki powiadomień

if pushover_user:  # sprawdź, czy zmienna PUSHOVER_USER w ogóle jest ustawiona
    if pushover_user.startswith("u"):  # identyfikatory Pushover userów zaczynają się od "u"
        print("Znaleziono użytkownika Pushover, wygląda poprawnie")
    else:
        print("Znaleziono użytkownika Pushover, ale nie zaczyna się od u")
else:
    print("Nie znaleziono użytkownika Pushover")

if pushover_token:  # sprawdź, czy zmienna PUSHOVER_TOKEN w ogóle jest ustawiona
    if pushover_token.startswith("a"):  # tokeny aplikacji Pushover zaczynają się od "a"
        print("Znaleziono token Pushover, wygląda poprawnie")
    else:
        print("Znaleziono token Pushover, ale nie zaczyna się od a")
else:
    print("Nie znaleziono tokena Pushover")

In [ ]:
# Zwykła funkcja Pythona wysyłająca powiadomienie push - identyczna logika jak w 1_lab1.pl.ipynb, niezależna od dostawcy LLM.
# payload buduje ciało zapytania POST zgodnie z API Pushover: user, token i treść wiadomości.
# requests.post() wykonuje samo wywołanie HTTP - wynik nie jest tu sprawdzany, to fallback używany tylko gdy e-mail nie jest skonfigurowany.
# print() pokazuje lokalnie, jaka wiadomość leci na telefon, zanim faktycznie zostanie wysłana.

def push(message):  # wysyła podaną wiadomość jako powiadomienie push
    print(f"Push: {message}")  # podgląd wiadomości w konsoli przed wysyłką
    payload = {"user": pushover_user, "token": pushover_token, "message": message}  # ciało zapytania zgodne z API Pushover
    requests.post(pushover_url, data=payload)  # wyślij powiadomienie push

In [ ]:
# Funkcja spinająca oba kanały komunikacji: e-mail jako główny, Pushover jako fallback.
# USE_EMAIL (z komórki wyżej) decyduje, którego kanału użyć - ta logika jest identyczna jak w oryginale, niezależna od dostawcy LLM.
# To ta funkcja (nie send_email ani push bezpośrednio) będzie użyta jako narzędzie Claude w dalszej części notatnika.

def send_message(subject, text_body, html_body):  # wyślij wiadomość głównym kanałem albo fallbackiem
    if USE_EMAIL:  # jeśli e-mail jest skonfigurowany
        send_email(subject, text_body, html_body)  # wyślij przez SMTP
    else:
        push(f"Subject: {subject}\n\n{text_body}")  # w przeciwnym razie wyślij push z tematem i treścią tekstową

### OK, teraz wszystko powinno działać!

In [ ]:
# Ręczne wywołanie testowe funkcji send_message() - sprawdza, że cały mechanizm wysyłki (e-mail albo push) działa końcowo-do-końca.
# Treść przetłumaczona na polski, bo to tekst, który faktycznie przeczytasz jako wiadomość testową.

send_message("Wielkie wieści", "Komunikacja działa!", "<html><body>Komunikacja <strong>działa!</strong></body></html>")  # test całego mechanizmu wysyłki

## Orkiestracja agentów

Istnieją 2 modele orkiestracji agentów: przez kod i przez LLM.

Przez kod: bardziej przewidywalne i deterministyczne.

Przez LLM: bardziej potężne.

Świetny opis tego zagadnienia (z perspektywy OpenAI Agents SDK) znajdziesz tutaj:

https://openai.github.io/openai-agents-python/multi_agent/

Zaczniemy od orkiestracji przez kod.

## Część 2: Orkiestracja przez kod

### Mechanizmy z `1_lab1.pl.ipynb`, rozszerzone o orkiestrację

Poniżej redefiniujemy `trace()` i `run()` dokładnie tak, jak w `1_lab1.pl.ipynb` (każdy notatnik tego kursu jest samodzielnym projektem) - z jedną zmianą: `run()` jest teraz `async def` i używa `AsyncAnthropic`, żeby `asyncio.gather()` mógł naprawdę wywołać kilka agentów naraz, a nie po kolei. Dokładamy też obsługę `tool_choice` (wymuszenie użycia narzędzia) i rozróżnienie zwykłego narzędzia od handoffu - oba mechanizmy pojawią się w Części 3.

In [ ]:
# Ta komórka definiuje trace() - dokładnie ten sam, lekki, lokalny odpowiednik obserwowalności co w 1_lab1.pl.ipynb.
# @contextmanager pozwala napisać funkcję generatorową, którą Python zamienia w obiekt obsługujący "with trace(...): ...".
# Kod przed yield wykonuje się przy wejściu do bloku with, kod po yield - przy wyjściu z niego, nawet jeśli w środku wystąpi wyjątek.
# Redefiniujemy ją tutaj, a nie importujemy z 1_lab1.pl.ipynb, bo każdy notatnik tego kursu jest samodzielnym, niezależnym projektem.
# W tym notatniku trace() opakuje sekwencje z asyncio.gather(), więc zobaczysz czas trwania CAŁEJ równoległej grupy wywołań, nie pojedynczego zapytania.

@contextmanager
def trace(name: str):  # lokalny, uproszczony odpowiednik trace() z OpenAI Agents SDK - bez wysyłki danych na zewnątrz
    start = time.time()  # zapamiętaj moment startu, żeby policzyć czas trwania
    print(f"[trace] start: {name}")  # znacznik początku sekwencji wywołań
    yield  # tutaj wykonuje się kod wewnątrz bloku "with trace(...):"
    print(f"[trace] koniec: {name} ({time.time() - start:.2f}s)")  # znacznik końca razem z czasem trwania

In [ ]:
# Ta komórka to async odpowiednik Agent + Runner.run() z 1_lab1.pl.ipynb, rozszerzony o dwa nowe mechanizmy potrzebne do orkiestracji.
# run() jest teraz "async def" i używa AsyncAnthropic (zainicjalizowanego w komórce z importami) - dzięki temu asyncio.gather() w Części 2 naprawdę uruchamia kilka wywołań naraz, a nie po kolei.
# Nowy parametr tool_choice pozwala wymusić użycie narzędzia (odpowiednik ModelSettings(tool_choice="required")) - potrzebne w Części 2 przy wysyłce e-maila.
# handle_tool_calls() rozróżnia teraz dwa rodzaje narzędzi: zwykłe (wynik wraca do modelu jako tool_result) i handoff (wynik kończy CAŁY run(), kontrola nie wraca) - oba pojawią się w Części 3.
# Ten jeden helper obsłuży całą resztę notatnika: orkiestrację przez kod (Część 2) i orkiestrację przez LLM, z narzędziami i z handoffs (Część 3).

async def handle_tool_calls(tool_use_blocks: list) -> tuple[list[dict] | None, str | None]:  # wykonuje wywołania narzędzi; rozróżnia zwykłe narzędzia od handoffów
    results = []  # lista bloków tool_result do wysłania z powrotem (pusta, jeśli trafimy na handoff)
    for block in tool_use_blocks:  # iteruj po każdym bloku tool_use z odpowiedzi
        tool = globals().get(block.name)  # znajdź funkcję Pythona o tej samej nazwie co narzędzie
        if tool is None:  # narzędzie o takiej nazwie nie istnieje w tym notatniku
            output = f"Nieznane narzędzie: {block.name}"  # komunikat błędu zwracany do Claude
        elif asyncio.iscoroutinefunction(tool):  # narzędzia typu agent-jako-tool/handoff są async (wywołują zagnieżdżone run())
            output = await tool(**block.input)  # wykonaj narzędzie asynchroniczne
        else:
            output = tool(**block.input)  # wykonaj zwykłe, synchroniczne narzędzie (np. send_email_tool)
        if tool is not None and getattr(tool, "is_handoff", False):  # sprawdź flagę ustawioną przez make_agent_tool(..., handoff=True)
            return None, output  # kontrola przechodzi do subagenta - jego tekst (output) kończy cały run(), reszta bloków w tej turze jest pomijana
        results.append({
            "type": "tool_result",  # Anthropic: blok tool_result zamiast wiadomości z rolą "tool" jak w OpenAI
            "tool_use_id": block.id,  # musi się zgadzać z id bloku tool_use, na który odpowiadamy
            "content": json.dumps(output),  # treść wyniku jako string JSON
        })
    return results, None  # zwracane bloki trafią razem do JEDNEJ wiadomości user; None oznacza "to nie był handoff"


async def run(instructions: str, user_message: str, history: list | None = None, tools: list | None = None, tool_choice: dict | None = None) -> tuple[str, list]:  # async odpowiednik Agent + Runner.run(), rozszerzony o tool_choice
    messages = (history or []) + [{"role": "user", "content": user_message}]  # doklej nową wiadomość do historii (albo zacznij od zera)
    kwargs = {"model": MODEL, "max_tokens": 16000, "system": instructions, "messages": messages, "tools": tools or []}  # wspólne argumenty obu wywołań .create() w tej pętli
    if tool_choice:  # tool_choice bywa None (domyślne zachowanie Claude) - dorzucamy klucz tylko, gdy faktycznie wymuszamy konkretny wybór
        kwargs["tool_choice"] = tool_choice  # np. {"type": "any"} wymusza użycie jakiegoś narzędzia
    response = await anthropic.messages.create(**kwargs)  # await, bo anthropic to teraz AsyncAnthropic
    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]  # wyciągnij bloki tool_use z odpowiedzi
        results, handoff_text = await handle_tool_calls(tool_use_blocks)  # wykonaj narzędzia; handoff_text != None oznacza przekazanie kontroli
        if handoff_text is not None:  # jeśli któreś narzędzie było handoffem
            return handoff_text, messages  # kontrola przeszła do subagenta - jego tekst kończy CAŁY run(), nie wraca jako tool_result
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user
        response = await anthropic.messages.create(**kwargs)  # kwargs["messages"] to ten sam obiekt messages, więc widzi dopisane wyżej wpisy
    text = next(block.text for block in response.content if block.type == "text")  # finalna odpowiedź tekstowa (content[0] bywa ThinkingBlock)
    messages.append({"role": "assistant", "content": response.content})  # zapisz finalną odpowiedź w historii do ewentualnego dalszego użycia
    return text, messages  # zwróć tekst (jak result.final_output) i historię (jak result.to_input_list())

In [ ]:
# Definicje trzech person sprzedażowych - każda dostaje ten sam kontekst biznesowy (intro), ale inny styl pisania.
# Te stringi pełnią rolę "instructions" agenta z OpenAI Agents SDK - w naszej wersji to po prostu prompt systemowy przekazywany do run().
# Treść promptów jest po polsku, bo to tekst czytany przez model (i pośrednio przez odbiorcę wygenerowanego e-maila).

intro = """
Jesteś przedstawicielem handlowym pracującym dla ComplAI,
firmy dostarczającej narzędzie SaaS zapewniające zgodność z SOC2 i przygotowanie do audytów, oparte na AI.
Piszesz e-maile.
"""  # wspólny kontekst biznesowy dla wszystkich trzech person

instructions1 = intro + "Twój styl pisania e-maili jest profesjonalny, poważny, z powagą i wiarygodnością."  # persona 1: profesjonalna
instructions2 = intro + "Twój styl pisania e-maili jest dowcipny, angażujący i pełen humoru."  # persona 2: humorystyczna
instructions3 = intro + "Twój styl pisania e-maili jest zwięzły, konkretny, w stylu zapracowanego seniorskiego menedżera."  # persona 3: kierownicza

In [ ]:
# W oryginale Agent(name=..., instructions=..., model=...) tworzy obiekt agenta z frameworka OpenAI Agents SDK.
# W naszej wersji "agent" to po prostu string instrukcji (system prompt) - dokładnie tak, jak jokester_instructions w 1_lab1.pl.ipynb.
# Zmienne sales_agent1/2/3 dostają nazwy takie same jak w oryginale, żeby reszta notatnika (Runner.run(sales_agent1, ...) -> run(sales_agent1, ...)) wyglądała maksymalnie podobnie.
# Nie ma tu żadnego wywołania API - to tylko przypisanie stringów do zmiennych.

sales_agent1 = instructions1  # "Professional Sales Agent" - w naszej wersji to sam string instrukcji
sales_agent2 = instructions2  # "Humorous Sales Agent"
sales_agent3 = instructions3  # "Executive Sales Agent"

In [ ]:
# Ta komórka pokazuje streaming - odpowiednik Runner.run_streamed() + stream_events() + ResponseTextDeltaEvent z OpenAI Agents SDK.
# Anthropic SDK ma do tego dedykowany context manager: anthropic.messages.stream(...), w wersji async używany z "async with" i "async for".
# stream.text_stream to generator kolejnych fragmentów tekstu - dokładnie to, co w oryginale filtrowało zdarzenia typu ResponseTextDeltaEvent.
# end="" i flush=True w print() sprawiają, że fragmenty tekstu doklejają się do siebie na bieżąco, zamiast czekać na całą odpowiedź.
# W przeciwieństwie do 1_lab1.pl.ipynb (klient synchroniczny), tu klient jest asynchroniczny, więc potrzebujemy "async with"/"async for".

async with anthropic.messages.stream(
    model=MODEL, max_tokens=16000, system=sales_agent1,
    messages=[{"role": "user", "content": "Napisz zimny e-mail sprzedażowy"}],
) as stream:  # context manager otwierający połączenie strumieniowe (wariant async)
    async for text in stream.text_stream:  # iteruj asynchronicznie po kolejnych fragmentach tekstu
        print(text, end="", flush=True)  # wypisz fragment bez nowej linii i bez buforowania

In [ ]:
# Trzy persony piszą e-mail równolegle - to jest "Orchestrating by Code": nasz kod (nie LLM) decyduje o kolejności i równoległości wywołań.
# asyncio.gather uruchamia trzy wywołania run() naraz, nie po kolei - każde to osobne zapytanie do Claude, ale wszystkie lecą jednocześnie.
# Działa to tylko dlatego, że run() jest zdefiniowane jako "async def" i używa AsyncAnthropic - synchroniczny klient blokowałby jedno wywołanie na raz.
# trace() z komórki wyżej opakowuje całą sekwencję jednym znacznikiem czasu w konsoli - lokalnym odpowiednikiem prawdziwego trace() z Agents SDK.
# Każde run() zwraca krotkę (tekst, historia) - tutaj interesuje nas tylko tekst, więc rozpakowujemy obie wartości i używamy tylko pierwszej.

message = "Napisz zimny e-mail sprzedażowy"  # to samo zadanie dla wszystkich trzech person

with trace("Równoległe zimne e-maile"):  # jeden znacznik czasu obejmujący wszystkie trzy równoległe wywołania
    results = await asyncio.gather(
        run(sales_agent1, message),  # persona profesjonalna
        run(sales_agent2, message),  # persona humorystyczna
        run(sales_agent3, message),  # persona kierownicza
    )  # asyncio.gather czeka, aż WSZYSTKIE trzy się skończą, i zwraca wyniki w tej samej kolejności

outputs = [answer for answer, _ in results]  # wyciągnij sam tekst odpowiedzi z każdej krotki (answer, history)

for output in outputs:  # wypisz każdy wygenerowany e-mail
    print(output + "\n\n")  # pusta linia między e-mailami dla czytelności

In [ ]:
# Agent-sędzia, który wybiera najlepszy z trzech wygenerowanych e-maili - kolejna persona, ten sam mechanizm co sales_agent1/2/3.
# decision to prompt systemowy tego agenta - po polsku, bo to tekst czytany przez model.
# sales_picker nie dostaje żadnych narzędzi - jego jedynym zadaniem jest ocena i wybór, nie akcja.

decision = """
Wybierasz najlepszy zimny e-mail sprzedażowy spośród podanych opcji.
Wyobraź sobie, że jesteś klientem i wybierz ten, na który najchętniej byś odpowiedział.
Nie podawaj wyjaśnienia; odpowiedz tylko wybranym e-mailem.
"""  # prompt systemowy agenta-sędziego

sales_picker = decision  # "Sales_picker" - w naszej wersji to sam string instrukcji, tak jak sales_agent1/2/3

In [ ]:
# Pełny przepływ "Orchestrating by Code": trzy e-maile równolegle, potem sekwencyjnie agent-sędzia wybiera najlepszy.
# Kolejność jest wymuszona przez sam kod Pythona (najpierw gather, potem osobne run() na sales_picker), nie przez decyzję LLM - stąd "by Code".
# emails łączy wszystkie trzy warianty w jeden string, który trafia jako user_message do sales_pickera.
# best to tekst zwrócony przez run() (druga wartość krotki to historia, tutaj pomijana przez _).

message = "Napisz zimny e-mail sprzedażowy"  # to samo zadanie co wcześniej

with trace("Przepływ wyboru e-maila"):  # jeden znacznik czasu obejmujący cały przepływ (generowanie + wybór)
    results = await asyncio.gather(
        run(sales_agent1, message),
        run(sales_agent2, message),
        run(sales_agent3, message),
    )  # trzy e-maile równolegle, tak jak w komórce wyżej
    outputs = [answer for answer, _ in results]  # wyciągnij sam tekst z każdej krotki

    emails = "Zimne e-maile sprzedażowe:\n\n" + "\n\nE-mail:\n\n".join(outputs)  # połącz wszystkie warianty w jeden string

    best, _ = await run(sales_picker, emails)  # sekwencyjnie: dopiero po zebraniu wszystkich trzech, poproś sędziego o wybór

    print(f"Najlepszy e-mail sprzedażowy:\n{best}")  # wypisz finalny wybór

## Nie ma tu prawdziwego "trace" do obejrzenia

Tak jak w `1_lab1.pl.ipynb`: nasz lokalny `trace()` wypisuje tylko dwie linie w konsoli (start/koniec + czas trwania) nad tą komórką - nie zapisuje niczego na `platform.openai.com/traces` ani żadnej innej platformie. To wystarczy, żeby zobaczyć, że trzy wywołania faktycznie leciały równolegle (czas całości będzie bliski czasowi NAJWOLNIEJSZEGO pojedynczego wywołania, nie sumie wszystkich trzech).

### Teraz dorzucimy do tego narzędzie.

In [ ]:
# Teraz to samo narzędzie send_message, ale w formacie wywoływanym przez Claude - odpowiednik dekoratora @function_tool z OpenAI Agents SDK.
# @function_tool w oryginale automatycznie generuje schemat JSON z sygnatury funkcji i docstringa (Args: ...) - w Anthropic nie ma takiej magii, schemat piszemy ręcznie.
# send_email_tool_json opisuje narzędzie: nazwę, opis czytany przez Claude i schemat argumentów (subject, text_body, html_body) w formacie JSON Schema.
# Sama funkcja send_email_tool wywołuje już istniejące send_message() (e-mail albo Pushover) i zwraca string ze statusem - Claude dostanie ten string jako tool_result.
# Nazwa funkcji musi się zgadzać z kluczem "name" w schemacie, bo nasza pętla run() szuka narzędzia po nazwie przez globals().get(block.name).

send_email_tool_json = {
    "name": "send_email_tool",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona
    "description": "Wyślij e-mail o podanym temacie i treści do wszystkich potencjalnych klientów sprzedażowych",  # opis czytany przez Claude
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI, i nie generuje go automatycznie z docstringa
        "type": "object",
        "properties": {
            "subject": {"type": "string", "description": "Temat e-maila"},
            "text_body": {"type": "string", "description": "Treść e-maila jako czysty tekst"},
            "html_body": {"type": "string", "description": "Treść e-maila w formacie HTML"},
        },
        "required": ["subject", "text_body", "html_body"],
        "additionalProperties": False,
    },
}


def send_email_tool(subject: str, text_body: str, html_body: str) -> str:  # wersja send_message() jako narzędzie Claude
    send_message(subject, text_body, html_body)  # wyślij e-mailem albo push, w zależności od USE_EMAIL
    return "E-mail wysłany pomyślnie"  # ten string trafi do Claude jako tool_result

### To osobny, ręcznie napisany schemat - bez automatycznego generowania z docstringa

W OpenAI Agents SDK dekorator `@function_tool` automatycznie generuje schemat JSON (boilerplate) z sygnatury funkcji i docstringa. W Anthropic nie ma takiej magii - schemat piszemy ręcznie, jak w `send_email_tool_json` w komórce wyżej. Zobacz go w akcji niżej.

In [ ]:
# Podgląd schematu JSON narzędzia, który Claude czyta przy decyzji, jak wypełnić argumenty.
# W OpenAI Agents SDK ten schemat (params_json_schema) jest generowany automatycznie z sygnatury funkcji i docstringa.
# W naszej wersji nie ma żadnej magii dekoratora - schemat to zwykły klucz "input_schema" w słowniku send_email_tool_json, wpisany ręcznie w komórce wyżej.

send_email_tool_json["input_schema"]  # schemat argumentów narzędzia (odpowiednik send_email_tool.params_json_schema)

In [ ]:
# Agent, który wybiera najlepszy e-mail I OD RAZU go wysyła, korzystając z narzędzia send_email_tool.
# require_tool wymusza użycie narzędzia - odpowiednik ModelSettings(tool_choice="required") z OpenAI Agents SDK.
# W Anthropic to samo osiąga się parametrem tool_choice={"type": "any"}: Claude MUSI użyć jakiegoś narzędzia, nie może tylko odpowiedzieć tekstem.
# sales_sender_tools to lista narzędzi dostępnych temu agentowi - tu tylko jedno, send_email_tool_json.

decision = """
Wybierasz najlepszy zimny e-mail sprzedażowy spośród podanych opcji.
Wyobraź sobie, że jesteś klientem i wybierz ten, na który najchętniej byś odpowiedział.
Następnie użyj swojego narzędzia, żeby wysłać ten e-mail.
"""  # zaktualizowany prompt - tym razem sędzia ma też wysłać wybrany e-mail

require_tool = {"type": "any"}  # Claude musi użyć jakiegoś narzędzia - odpowiednik ModelSettings(tool_choice="required")

sales_sender = decision  # "Sales Sender" - string instrukcji
sales_sender_tools = [send_email_tool_json]  # lista narzędzi dostępnych temu agentowi

In [ ]:
# Pełny przepływ z wysyłką: trzy e-maile równolegle, potem sales_sender wybiera najlepszy I wysyła go narzędziem send_email_tool.
# tools=sales_sender_tools i tool_choice=require_tool trafiają do run() jako dodatkowe argumenty - to włącza pętlę tool-use wewnątrz run().
# response to tekst zwrócony przez run() po zakończeniu pętli (druga wartość krotki to historia, tutaj pomijana przez _).

message = "Napisz zimny e-mail sprzedażowy"  # to samo zadanie co wcześniej

with trace("Przepływ wyboru e-maila z wysyłką"):  # jeden znacznik czasu obejmujący cały przepływ
    results = await asyncio.gather(
        run(sales_agent1, message),
        run(sales_agent2, message),
        run(sales_agent3, message),
    )  # trzy e-maile równolegle
    outputs = [answer for answer, _ in results]  # wyciągnij sam tekst z każdej krotki

    emails = "Zimne e-maile sprzedażowe:\n\n" + "\n\nE-mail:\n\n".join(outputs)  # połącz wszystkie warianty w jeden string

    response, _ = await run(sales_sender, emails, tools=sales_sender_tools, tool_choice=require_tool)  # wybierz i wyślij najlepszy

    print(f"Finalna odpowiedź:\n{response}")  # wypisz odpowiedź agenta po wysłaniu

### Czy to zadziałało?!

Sprawdź output powyżej i swoją skrzynkę (albo powiadomienie push). Tak jak wyżej - nie ma tu prawdziwej platformy trace do obejrzenia, tylko lokalny znacznik czasu w konsoli. Mniejsze modele mogą wymagać więcej czasu i eksperymentowania, żeby użycie narzędzia było niezawodne.

## Część 3: Orkiestracja przez LLM

### 3a: przez narzędzia

Najprostszy sposób, żeby jeden Agent zdecydował się wywołać innego, to potraktowanie go jak wywołanie narzędzia.

OpenAI Agents SDK daje bardzo prosty sposób, żeby to zrobić.

To działa najlepiej, gdy przepływ wygląda tak:

Agent A -> Agent B -> Agent A

I w klasycznej sytuacji "Agenta Planującego".

### `agent.as_tool()` z OpenAI Agents SDK nie ma odpowiednika w Anthropic - budujemy własny

`agent.as_tool(tool_name=..., tool_description=...)` w oryginale automatycznie zamienia dowolnego agenta w narzędzie: wywołanie tego narzędzia uruchamia agenta na podanym wejściu i zwraca jego finalną odpowiedź jako wynik narzędzia. Poniżej `make_agent_tool()` robi dokładnie to samo ręcznie, na `run()` zdefiniowanym wyżej.

In [ ]:
# Odpowiednik agent.as_tool(...) z OpenAI Agents SDK - zamienia innego agenta (jego instrukcje) w narzędzie wywoływane przez agenta nadrzędnego.
# make_agent_tool() zwraca parę: schemat JSON opisujący narzędzie dla Claude i funkcję Pythona, która przy wywołaniu uruchamia PEŁNY, zagnieżdżony run() na subagencie.
# Zwrócony tekst subagenta trafia z powrotem do agenta nadrzędnego jako zwykły tool_result - to jest sedno "agent jako narzędzie": kontrola wraca (A -> B -> A).
# Parametr handoff (domyślnie False) przyda się dopiero w sekcji o handoffs niżej - ta sama funkcja posłuży tam do zbudowania innej semantyki (kontrola NIE wraca, A -> B).
# Parametr tool_choice przechodzi bez zmian do zagnieżdżonego run() - potrzebny np. gdy subagent (jak sales_sender) MUSI użyć własnego narzędzia, żeby coś faktycznie zrobił, a nie tylko opisał.

def make_agent_tool(name: str, description: str, instructions: str, tools: list | None = None, tool_choice: dict | None = None, handoff: bool = False):  # buduje parę (schemat, funkcja) z instrukcji subagenta
    async def agent_tool(input: str) -> str:  # funkcja narzędzia - wywołuje subagenta i zwraca jego finalny tekst
        text, _ = await run(instructions, input, tools=tools, tool_choice=tool_choice)  # pełne, zagnieżdżone wywołanie run() na subagencie, z tym samym tool_choice co u samego subagenta
        return text  # tekst subagenta trafia do Claude jako wynik narzędzia (albo jako finalna odpowiedź run(), jeśli handoff=True)
    agent_tool.is_handoff = handoff  # oznacz funkcję flagą - handle_tool_calls() sprawdza ją, żeby rozróżnić zwykłe narzędzie od handoffu
    schema = {
        "name": name,  # nazwa narzędzia - musi się zgadzać z nazwą, pod którą zapiszesz zwróconą funkcję w globals()
        "description": description,  # opis czytany przez Claude przy decyzji, kiedy użyć tego narzędzia
        "input_schema": {
            "type": "object",
            "properties": {
                "input": {"type": "string", "description": "Instrukcja dla subagenta, np. treść zadania do wykonania"},
            },
            "required": ["input"],
            "additionalProperties": False,
        },
    }
    return schema, agent_tool  # para gotowa do wpisania do globals() (funkcja) i do listy tools= (schemat)

In [ ]:
# Pojedyncza demonstracja mechanizmu make_agent_tool() - zamiana sales_agent1 w narzędzie, odpowiednik sales_agent1.as_tool(tool_name=..., tool_description=...).
# tool1_json trafi do listy tools= w kolejnych komórkach; sales_email_writer_1 musi być zmienną globalną o tej samej nazwie co "name" w schemacie.
# Właściwe użycie (razem z pozostałymi dwiema personami) jest w kolejnej komórce - ta tutaj tylko pokazuje wynik pojedynczo.

description = "Użyj tego narzędzia, żeby napisać e-mail sprzedażowy. W argumencie input po prostu poinstruuj je, żeby napisało e-mail sprzedażowy."  # opis czytany przez Claude

tool1_json, sales_email_writer_1 = make_agent_tool("sales_email_writer_1", description, sales_agent1)  # zbuduj parę (schemat, funkcja) z persony 1
tool1_json  # podgląd schematu narzędzia (odpowiednik podglądu obiektu tool1 z oryginału)

### Teraz możemy zebrać wszystkie narzędzia razem:

Po jednym narzędziu dla każdego z naszych 3 agentów piszących e-maile

I narzędzie dla naszej funkcji wysyłającej e-maile

In [ ]:
# Zbieramy wszystkie narzędzia w jedną listę: trzy narzędzia piszące e-maile (po jednym na personę) i jedno narzędzie do wysyłki.
# Każda para (schemat, funkcja) z make_agent_tool() musi trafić do globals() pod nazwą zgodną z "name" w schemacie - stąd przypisanie do zmiennych sales_email_writer_1/2/3.
# tools to lista samych schematów JSON, przekazywana potem jako tools= do run() - dokładnie tak samo jak notifier_tools czy twin_tools we wcześniejszych notatnikach.

tool1_json, sales_email_writer_1 = make_agent_tool("sales_email_writer_1", description, sales_agent1)  # narzędzie piszące e-mail w stylu profesjonalnym
tool2_json, sales_email_writer_2 = make_agent_tool("sales_email_writer_2", description, sales_agent2)  # narzędzie piszące e-mail w stylu humorystycznym
tool3_json, sales_email_writer_3 = make_agent_tool("sales_email_writer_3", description, sales_agent3)  # narzędzie piszące e-mail w stylu kierowniczym

tools = [tool1_json, tool2_json, tool3_json, send_email_tool_json]  # komplet narzędzi dostępnych agentowi-menedżerowi niżej

tools  # podgląd listy schematów

## A teraz czas na naszego Menedżera Sprzedaży - naszego agenta planującego

In [ ]:
# Agent-menedżer, który samodzielnie decyduje, KIEDY i W JAKIEJ KOLEJNOŚCI wywołać narzędzia - to jest "Orchestrating by LLMs", w przeciwieństwie do Części 2, gdzie kolejność wymuszał nasz kod Pythona.
# instructions to krótki opis roli, task to szczegółowa instrukcja krok po kroku, przekazywana jako user_message do run().
# sales_manager (podobnie jak wcześniejsze persony) to w naszej wersji zwykły string instrukcji - "agentem" czyni go dopiero połączenie z listą tools w wywołaniu run().

instructions = """
Jesteś Menedżerem Sprzedaży w ComplAI. Twoim celem jest znalezienie najlepszego zimnego e-maila sprzedażowego, korzystając z narzędzi sales_writer.
"""  # rola agenta-menedżera

task = """
Wykonaj następujące kroki:

1. Wygeneruj wersje robocze: użyj każdego z trzech narzędzi sales_email_writer, żeby wygenerować różne wersje e-maila.
Po prostu poinstruuj każde z nich, żeby napisało e-mail sprzedażowy; nie są potrzebne dalsze szczegóły.
Nie kontynuuj, dopóki wszystkie trzy wersje nie będą gotowe, po jednej z każdego narzędzia.

2. Oceń i wybierz: przejrzyj wersje robocze i wybierz jeden, najlepszy e-mail, kierując się własnym osądem, który będzie najskuteczniejszy.

3. Użyj swojego narzędzia, żeby wysłać najlepszy e-mail (i tylko najlepszy) do użytkownika. Wyślij tylko 1 e-mail.
"""  # szczegółowa instrukcja krok po kroku dla agenta-menedżera

sales_manager = instructions  # "Sales Manager" - string instrukcji

In [ ]:
# draw_graph() z OpenAI Agents SDK rysuje graf agentów/narzędzi przez graphviz - nie ma tu bezpośredniego odpowiednika w Anthropic ani w naszym run().
# Zamiast dorzucać nową zależność (graphviz) tylko dla jednej wizualizacji, piszemy prosty, tekstowy odpowiednik: drzewo zależności wypisane w konsoli.
# fn = globals().get(...) sprawdza, czy dane narzędzie ma ustawioną flagę is_handoff (patrz make_agent_tool()) - przyda się dopiero w wersji z handoffami niżej, tu wszystkie gałęzie będą zwykłymi narzędziami.
# To świadome uproszczenie, tak jak lokalny trace() w 1_lab1.pl.ipynb - nie prawdziwy graf wektorowy, tylko czytelny szkic tej samej informacji.
# Ta sama funkcja zostanie użyta ponownie w sekcji o handoffs, żeby pokazać różnicę w etykietach gałęzi.

def print_agent_graph(name: str, tools: list[dict]) -> None:  # tekstowy odpowiednik draw_graph() - drzewo agent -> narzędzia/handoffy
    print(name)  # nazwa agenta na górze drzewa
    for tool in tools:  # iteruj po schematach narzędzi/handoffów tego agenta
        fn = globals().get(tool["name"])  # znajdź powiązaną funkcję, żeby sprawdzić flagę is_handoff
        label = "handoff" if getattr(fn, "is_handoff", False) else "narzędzie"  # rozróżnij typ gałęzi
        print(f"  └── [{label}] {tool['name']}")  # gałąź drzewa z etykietą typu

print_agent_graph("Sales Manager", tools)  # podgląd grafu: menedżer -> trzy narzędzia-persony + narzędzie wysyłki

In [ ]:
# Uruchomienie agenta-menedżera - Claude sam decyduje, kiedy wywołać które narzędzie (Orchestrating by LLMs).
# tools=tools zawiera komplet czterech narzędzi zebranych wyżej; task to szczegółowa instrukcja krok po kroku.
# result to krotka (tekst, historia) zwrócona przez run() - w oryginale result.final_output, tutaj result[0].

with trace("Menedżer sprzedaży"):  # jeden znacznik czasu obejmujący całą pracę menedżera
    result = await run(sales_manager, task, tools=tools)  # menedżer sam decyduje o kolejności wywołań narzędzi

## Sprawdź swój e-mail (i folder Spam)!

Nie ma tu prawdziwego trace do sprawdzenia (patrz uwaga wyżej) - ale koniecznie sprawdź skrzynkę, w tym folder Spam/Junk - w końcu to w zasadzie wiadomość spamowa..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Niepewne wyniki?</h2>
            <span style="color:#ff7800;">To normalka w świecie Agentic AI, a zwłaszcza przy orkiestracji przez LLM.
            Żeby to rozwiązać, będziesz musiał eksperymentować i iterować na promptach. Zwłaszcza przy mniejszych modelach, może być potrzeba
            więcej eksperymentów, żeby uzyskać niezawodne wyniki.
            </span>
        </td>
    </tr>
</table>

## Część 3: Orkiestracja przez LLM

### 3b: przez handoffs

Nie jestem fanem handoffs. Wydają się bardzo zawodne. Nie są używane konsekwentnie przez inne frameworki.

W tle, OpenAI Agents SDK i tak zaimplementował je przez narzędzia.

### Handoffs to sposób, w jaki agent może zlecić zadanie innemu agentowi, przekazując mu kontrolę

Handoffs i agent-jako-narzędzie są podobne:

W obu przypadkach jeden Agent może współpracować z innym Agentem

Przy narzędziach kontrola wraca

A -> B -> A

Przy handoffs kontrola przechodzi na drugą stronę

A -> B

In [ ]:
# Wersja z handoffem zamiast zwykłego narzędzia na końcu - menedżer generuje trzy wersje robocze, a potem PRZEKAZUJE KONTROLĘ (nie zleca zadanie) agentowi sales_sender.
# Różnica względem agent-jako-narzędzie: przy zwykłym narzędziu kontrola wraca do menedżera (A -> B -> A), przy handoffie kontrola przechodzi na stałe (A -> B) - menedżer już nie decyduje, co dalej.
# sales_sender_handoff budujemy TĄ SAMĄ funkcją make_agent_tool() co wcześniej, tylko z handoff=True - to ustawia flagę is_handoff, którą sprawdza nasza pętla w run().
# tool_choice=require_tool jest tu równie ważne jak przy zwykłym narzędziu (komórka z sales_sender wyżej) - bez tego Claude wewnątrz handoffu mógłby tylko OPISAĆ, co by wysłał, zamiast faktycznie wywołać send_email_tool.
# tools tej wersji menedżera to tylko trzy narzędzia piszące e-maile (bez send_email_tool) - wysyłką zajmuje się już subagent sales_sender po przejęciu kontroli.

instructions = """
Jesteś Menedżerem Sprzedaży w ComplAI. Zlecasz swojemu zespołowi sprzedaży napisanie e-maili, a potem przekazujesz je wszystkie agentowi wybierającemu.
"""  # zaktualizowana rola - menedżer już nie wybiera ani nie wysyła sam

task = """
Wykonaj następujące kroki:

1. Wygeneruj wersje robocze: użyj każdego z trzech narzędzi sales_email_writer, żeby wygenerować różne wersje e-maila.
Po prostu poinstruuj każde z nich, żeby napisało e-mail sprzedażowy; nie są potrzebne dalsze szczegóły.
Nie kontynuuj, dopóki wszystkie trzy wersje nie będą gotowe, po jednej z każdego narzędzia.

2. Przekaż kontrolę agentowi wysyłającemu, żeby wybrał i wysłał najlepszy e-mail.
"""  # zaktualizowana instrukcja - krok 2 to teraz handoff, nie wybór + wysyłka przez menedżera

handoff_description = "Przekaż wybór i wysyłkę najlepszego e-maila agentowi wysyłającemu, razem z trzema wygenerowanymi wersjami roboczymi"  # opis czytany przez Claude

sales_sender_handoff_json, sales_sender_handoff = make_agent_tool(
    "sales_sender_handoff", handoff_description, sales_sender, tools=sales_sender_tools, tool_choice=require_tool, handoff=True,
)  # ta sama funkcja co przy agent-jako-narzędzie, tylko handoff=True zmienia semantykę zakończenia pętli w run(), a tool_choice wymusza faktyczną wysyłkę

tools = [tool1_json, tool2_json, tool3_json, sales_sender_handoff_json]  # trzy narzędzia piszące + jedno narzędzie-handoff (bez send_email_tool - tym zajmie się subagent po przejęciu kontroli)

sales_manager = instructions  # "Sales Manager" - string instrukcji (nowa wersja, bez samodzielnego wysyłania)

In [ ]:
# Odpowiednik draw_graph() dla wersji z handoffem - reużywamy tę samą funkcję print_agent_graph() z komórki wyżej.
# Tym razem menedżer ma trzy zwykłe narzędzia i jeden handoff - flaga is_handoff na sales_sender_handoff sprawi, że ta gałąź dostanie inną etykietę.

print_agent_graph("Sales Manager", tools)  # podgląd grafu wersji z handoffem: menedżer -> 3 narzędzia + 1 handoff

In [ ]:
# Uruchomienie menedżera w wersji z handoffem - Claude generuje trzy wersje, a potem wywołuje handoff, co kończy run() tekstem subagenta sales_sender.
# result to krotka (tekst, historia) - tekst pochodzi już od sales_sender (po przejęciu kontroli), nie od sales_manager.

with trace("Menedżer sprzedaży"):  # jeden znacznik czasu obejmujący całą pracę menedżera i subagenta po handoffie
    result = await run(sales_manager, task, tools=tools)  # menedżer generuje wersje, potem przekazuje kontrolę przez handoff

### Sprawdź swój e-mail!!

Tak jak wyżej - nie ma tu prawdziwej platformy trace, tylko lokalny znacznik czasu w konsoli powyżej.

Pamiętaj, że handoffs (i ich uproszczony odpowiednik w tym notatniku) bywają zawodne i trochę frustrujące. W oryginale trzeba było wymusić użycie narzędzia, żeby to w ogóle zadziałało. Jeśli nie dostajesz niezawodnego zachowania, spróbuj iterować na promptach - albo użyj większego modelu. I ciesz się procesem; o to właśnie chodzi w Agentic AI!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">Spróbuj rozszerzyć to o dodatkową orkiestrację - może kolejną iterację e-maila, żeby go dopracować.
            Użyj orkiestracji przez kod i orkiestracji przez LLM.
            TRUDNE WYZWANIE: przejdź na profesjonalnego dostawcę e-mail, jak SendGrid albo Resend. Potem obsłuż odpowiedź użytkownika.
            A potem spraw, żeby SDR odpowiadał, podtrzymując rozmowę!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implikacje komercyjne</h2>
            <span style="color:#00bfff;">To ma natychmiastowe zastosowanie w automatyzacji sprzedaży; ale bardziej ogólnie może to być zastosowane
            do automatyzacji end-to-end dowolnego procesu biznesowego przez rozmowy i narzędzia. Pomyśl, jak mógłbyś zastosować takie
            rozwiązanie agentowe w swojej codziennej pracy.
            </span>
        </td>
    </tr>
</table>